# SMS Spam Classifier

#### This is an SMS Spam Classifier that predicts whether a text is spam or not and shows the confident score of that prediction

### To properly complete this project, we will divide this model into parts

## Part 1: Importing and Understading the Data

In [3]:
#we will first import the libraries

import re
import pandas as pd
import matplotlib as plt

In [4]:
#the we will load the csv dataset named collection

collection = pd.read_csv('SMSSpamCollection', sep='\t', names=['label', 'message'])

In [5]:
#always peep through the data to see how it looks like

collection.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
#we also look at the shape to know 

collection.shape

(5572, 2)

In [7]:
#counting the amount of ham and spam

collection['label'].value_counts()

label
ham     4825
spam     747
Name: count, dtype: int64

## Part 2: Cleaning the data

#### Now because the machine doesn't naturally understand nlp, will turn the text into an easier nlp model

In [8]:
#first we lowercase it

collection['message'] = collection['message'].str.lower()

In [9]:
# we peep into the data to see what has changed

collection.head()

,label,message
0,ham,"go until jurong point, crazy.. available only ..."
1,ham,ok lar... joking wif u oni...
2,spam,free entry in 2 a wkly comp to win fa cup fina...
3,ham,u dun say so early hor... u c already then say...
4,ham,"nah i don't think he goes to usf, he lives aro..."


In [10]:
# we create a function for easier cleaning, that is to remove punctuation and noise

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text 

#### the ^ in the sub means not

In [11]:
# then we apply it to the massages again

collection['message'] = collection['message'].apply(clean_text)

In [12]:
print(collection['message'])

0       go until jurong point crazy available only in ...
1                                 ok lar joking wif u oni
2       free entry in  a wkly comp to win fa cup final...
3             u dun say so early hor u c already then say
4       nah i dont think he goes to usf he lives aroun...
                              ...                        
5567    this is the nd time we have tried  contact u u...
5568                   will  b going to esplanade fr home
5569    pity  was in mood for that soany other suggest...
5570    the guy did some bitching but i acted like id ...
5571                            rofl its true to its name
Name: message, Length: 5572, dtype: str


## Part 3: Training and Testing the Model

### To train the model, we give the model training data and testing the data

#### To train the data we we will declare x and y (x meaning input and y meaning target)

In [13]:
# declaring the x and y

x = collection['message']

y = collection['label']

### For us to training both x and y, we will need to split the data

### For that to happen we  will use the scikit learn library 

In [14]:
from sklearn.model_selection import train_test_split

In [15]:
# we use the train_test_split() to train and test the data

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

#### the stratify preserves the data after the split

In [16]:
# we look the shape of the training and testing for x 

print(x_train.shape)
print(x_test.shape)

(4457,)
(1115,)


In [17]:
# we look the shape of the training and testing for y

print(y_train.shape)
print(y_test.shape)

(4457,)
(1115,)


#### So it's like this:
#### x_train → training messages
#### y_train → answers for training messages

#### x_test  → unseen messages
#### y_test  → correct answers for those messages

In [18]:
print(y_train.value_counts())
print(y_test.value_counts())

label
ham     3859
spam     598
Name: count, dtype: int64
label
ham     966
spam    149
Name: count, dtype: int64


## Part 3.5 Turning Data into Numbers

#### To properly train this model, we need to know how many times a text appears

### For this we will use a Bag of Words

In [19]:
# to use BoW, we will use the CountVectorizer

from sklearn.feature_extraction.text import CountVectorizer

In [20]:
# you will then pass the CountVectorizer model through vectorizer

vectorizer = CountVectorizer()

In [21]:
# It converts the text into numbers and counts how often each word occurs
# the fit_ learns the data in x_train

x_train_bow = vectorizer.fit_transform(x_train)

In [22]:
# no fit_ because we don't want the model to know the data, just know is numbers

x_test_bow = vectorizer.transform(x_test)

In [23]:
# we look at their shape

print(x_train_bow.shape)

print(x_test_bow.shape)


(4457, 7531)
(1115, 7531)


#### this means 4,457/1,115 (SMS) messages and 7531 different words/features

In [24]:
# this is to know the names the model has already learnt

print(vectorizer.get_feature_names_out()[:20])

['aa' 'aah' 'aaniye' 'aathilove' 'aathiwhere' 'ab' 'abbey' 'abdomen'
 'abeg' 'abelu' 'aberdeen' 'abi' 'ability' 'abiola' 'abj' 'able'
 'abnormally' 'about' 'aboutas' 'above']


#### This is basically how the model goes through:

#### SMS messages
     ↓
#### CountVectorizer
     ↓
#### words → numbers
     ↓
#### Naive Bayes
     ↓
#### learns relationship between words and spam/ham
     ↓
#### predict new SMS

In [25]:
# to see how the actual numbers look like

print(x_train_bow[0].toarray())

[[0 0 0 ... 0 0 0]]


#### So from the model we have, we have turned every string to numbers. Now through this numbers we can then use the Naive Bayes Model to check whether the number is spam or not

### Part 4: Naive Bayes Model

In [26]:
# to use the nbm, we import it using;

from sklearn.naive_bayes import MultinomialNB

#### Multinomial Naive Bayes is designed to work well with this kind of discrete feature data, which makes it a classic choice for text classification. Because of the number count

In [27]:
# we load the model

model = MultinomialNB()

In [28]:
# we do the learning where x_train_bow contains the numerical representation and y_train contains the correct answer

model.fit(x_train_bow, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[3859., 598.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.14,-2.01]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[<U4](2,)","['ham','spam']"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 7531)","[[1.,3.,1.,...,1.,1.,1.], [0.,0.,0.,...,1.,0.,0.]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 7531)","[[-10.25, -9.55,-10.25,...,-10.25,-10.25,-10.25], [ -9.88, -9.88, -9.88,..., -9.19, -9.88, -9.88]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,7531


#### This means 'Look at these numerical SMS features and their correct labels, and learn the relationship between them.'

In [29]:
#now we give the model data it hasn't trained on

y_pred = model.predict(x_test_bow)

## Part 5: Measuring the Accuracy

In [30]:
# so after the predictions let's measure it's accuracy

from sklearn.metrics import accuracy_score, classification_report

In [31]:
# to know the accuracy, we will pass it through the y_test and y_pred

accuracy = accuracy_score(y_test, y_pred)

In [32]:
print(f"Accuracy: {accuracy}")

Accuracy: 0.9820627802690582


In [33]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       966
        spam       0.98      0.89      0.93       149

    accuracy                           0.98      1115
   macro avg       0.98      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115



#### The central ML workflow for this project 

#### X_train + y_train
    ↓
####       .fit()
    ↓
####   LEARN PATTERNS
    ↓
####       X_test
    ↓
####     .predict()
    ↓
####     PREDICTIONS
    ↓
####  compare with y_test
    ↓
####      EVALUATION

### First, look at the result again

In [34]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       966
        spam       0.98      0.89      0.93       149

    accuracy                           0.98      1115
   macro avg       0.98      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115



#### So from this we can say that our model can detect all ham messages but it can't not detect all spam messages as 149 * 0.89 = 133

#### Because your dataset contains many more ham messages, that model could still achieve surprisingly high accuracy.

#### That's why we're looking at:

### Precision, Recall and F1

#### So the classification goes like this:

####              SPAM

#### Precision ───────────────► 0.98
#### "Are my spam predictions correct?"

#### Recall ──────────────────► 0.89
#### "Did I catch the actual spam?"

#### F1 ──────────────────────► 0.93
#### "How well do I balance both?"

### Now since we know that it's recall 0.89, we can check which spam is it getting wrong

In [35]:
from sklearn.metrics import confusion_matrix

In [36]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[963   3]
 [ 17 132]]


In [37]:
# we create a Dataframe to find
import pandas as pd

results = pd.DataFrame({
    'message': x_test, 
    'actual': y_test,
    'predicted': y_pred
})

In [38]:
# the SMS messages is x_test
# the actual labels is y_test
# the model's predictions is y_pred

In [39]:
#we peek into the new frame done

results

,message,actual,predicted
2825,no need to buy lunch for me i eat maggi mee,ham,ham
3695,ok im not sure what time i finish tomorrow but...,ham,ham
3904,waiting in e car my mum lor u leh reach home ...,ham,ham
576,you have won cash or a prize to claim call,spam,spam
2899,if you r home then come down within min,ham,ham
...,...,...,...
854,ah poor babyhope urfeeling bettersn luv probth...,ham,ham
5044,o ic lol should play doors sometime yo,ham,ham
2015,ambrithmaduraimet u in arun dha marrgeremembr,ham,ham
3380,dear umma she called me now,ham,ham


#### To bring out the wrong labels (false negatives), we create another dataframe

In [40]:
false_negatives = results[
    (results['actual'] == 'spam')  &  (results['predicted'] == 'ham')
]

In [41]:
false_negatives

,message,actual,predicted
5,freemsg hey there darling its been weeks now ...,spam,ham
3981,ringtoneking,spam,ham
3360,sorry i missed your call lets talk when you ha...,spam,ham
5449,latest news police station toilet stolen cops ...,spam,ham
1269,can u get phone now i wanna chat set up meet...,spam,ham
3064,hi babe its jordan how r u im home from abroad...,spam,ham
1940,more people are dogging in your area now call ...,spam,ham
1430,for sale arsenal dartboard good condition but...,spam,ham
731,email alertfrom jeri stewartsize kbsubject low...,spam,ham
2804,freemsgfav xmas tonesreply real,spam,ham


#### To get a more accurate result, we will use the TdidfVectorizer

In [42]:
# we will first load it

from sklearn.feature_extraction.text import TfidfVectorizer

In [43]:
# pass it through a container

new_vectorizer = TfidfVectorizer()

In [44]:
# use the container to train the model

x_train_tdidf = new_vectorizer.fit_transform(x_train)
x_test_tdidf = new_vectorizer.transform(x_test)

In [45]:
new_model = MultinomialNB()

In [46]:
new_model.fit(x_train_tdidf, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[3859., 598.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.14,-2.01]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[<U4](2,)","['ham','spam']"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 7531)","[[0.32,1.22,0.1 ,...,0.28,0.35,0.17], [0. ,0. ,0. ,...,0.31,0. ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 7531)","[[-9.6 ,-9.08,-9.78,...,-9.63,-9.57,-9.72], [-9.2 ,-9.2 ,-9.2 ,...,-8.93,-9.2 ,-9.2 ]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,7531


In [47]:
new_y_pred = new_model.predict(x_test_tdidf)

#### After the new prediction, we can check for the new classification report and accuracy

In [48]:
#using the following

accuracy = accuracy_score(y_test, new_y_pred)

print(f"Accuracy: {accuracy}")
print(classification_report(y_test, new_y_pred))

Accuracy: 0.9542600896860987
              precision    recall  f1-score   support

         ham       0.95      1.00      0.97       966
        spam       1.00      0.66      0.79       149

    accuracy                           0.95      1115
   macro avg       0.97      0.83      0.88      1115
weighted avg       0.96      0.95      0.95      1115



In [49]:
cm = confusion_matrix(y_test, new_y_pred)

print(cm)

[[966   0]
 [ 51  98]]


#### So from this first two experiments, we have found an interesting thing 

### Bag of Words + Naive Bayes does better than TD-idF + Naive Bayes (even tho Td-idF places more importance on the words)

#### But we want to increase the recall, 

### so let's change the model to Linear Regression

In [50]:
# so first we will load the model

from sklearn.linear_model import LogisticRegression

In [51]:
lr_model = LogisticRegression()

In [52]:
lr_model.fit(x_train_bow, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [53]:
lr_y_pred = lr_model.predict(x_test_bow)

In [54]:
# check the new result

print(f"Accuracy: {accuracy_score(y_test, lr_y_pred)}")
print(classification_report(y_test, lr_y_pred))

Accuracy: 0.9829596412556054
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       966
        spam       1.00      0.87      0.93       149

    accuracy                           0.98      1115
   macro avg       0.99      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [55]:
print(confusion_matrix(y_test, lr_y_pred))

[[966   0]
 [ 19 130]]


In [56]:
new_lr_model = LogisticRegression()

In [57]:
new_lr_model.fit(x_train_tdidf, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default sol

In [58]:
new_lr_y_pred = new_lr_model.predict(x_test_tdidf)

In [59]:
# check the new result

print(f"Accuracy: {accuracy_score(y_test, new_lr_y_pred)}")
print(classification_report(y_test, new_lr_y_pred))
print(confusion_matrix(y_test, new_lr_y_pred))

Accuracy: 0.9704035874439462
              precision    recall  f1-score   support

         ham       0.97      1.00      0.98       966
        spam       1.00      0.78      0.88       149

    accuracy                           0.97      1115
   macro avg       0.98      0.89      0.93      1115
weighted avg       0.97      0.97      0.97      1115

[[966   0]
 [ 33 116]]


In [60]:
print(confusion_matrix(y_test, new_lr_y_pred))

[[966   0]
 [ 33 116]]


#### But before that, we will learn class_weights.

#### We will pass 'balanced' feature due to the difference in ham v spam (as ham > spam)

In [61]:
balanced_lr_model = LogisticRegression(class_weight='balanced')

In [62]:
balanced_lr_model.fit(x_train_bow, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good defau

In [63]:
balanced_y_pred = balanced_lr_model.predict(x_test_bow)

In [64]:
print(f"Accuracy: {accuracy_score(y_test, balanced_y_pred)}")
print(classification_report(y_test, balanced_y_pred))
print(confusion_matrix(y_test, balanced_y_pred))

Accuracy: 0.9874439461883409
              precision    recall  f1-score   support

         ham       0.99      1.00      0.99       966
        spam       0.99      0.91      0.95       149

    accuracy                           0.99      1115
   macro avg       0.99      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115

[[965   1]
 [ 13 136]]


So after improving our model to 91%, we want to know which of the model is it still getting wrong

In [65]:
results = pd.DataFrame({
        'messages': x_test,
        'actual': y_test,
        'predicted': balanced_y_pred
})

results

,messages,actual,predicted
2825,no need to buy lunch for me i eat maggi mee,ham,ham
3695,ok im not sure what time i finish tomorrow but...,ham,ham
3904,waiting in e car my mum lor u leh reach home ...,ham,ham
576,you have won cash or a prize to claim call,spam,spam
2899,if you r home then come down within min,ham,ham
...,...,...,...
854,ah poor babyhope urfeeling bettersn luv probth...,ham,ham
5044,o ic lol should play doors sometime yo,ham,ham
2015,ambrithmaduraimet u in arun dha marrgeremembr,ham,ham
3380,dear umma she called me now,ham,ham


In [66]:
false_negative = results [
    (results['actual'] == 'spam') & (results['predicted'] == 'ham')
]

In [67]:
false_negative

,messages,actual,predicted
3981,ringtoneking,spam,ham
3360,sorry i missed your call lets talk when you ha...,spam,ham
5449,latest news police station toilet stolen cops ...,spam,ham
1460,bought one ringtone and now getting texts cost...,spam,ham
1430,for sale arsenal dartboard good condition but...,spam,ham
731,email alertfrom jeri stewartsize kbsubject low...,spam,ham
2804,freemsgfav xmas tonesreply real,spam,ham
607,xclusiveclubsaisai morow soiree speciale zouk...,spam,ham
2823,romcapspam everyone around should be respondin...,spam,ham
751,do you realize that in about years well have ...,spam,ham


Since BoW can't do it, since 

In [68]:
from sklearn.feature_extraction.text import CountVectorizer

ngram_vectorizer = CountVectorizer(ngram_range=(1, 2))

In [69]:
ngram_x_train = ngram_vectorizer.fit_transform(x_train)

ngram_x_test = ngram_vectorizer.transform(x_test)

In [70]:
balanced_lr_model.fit(ngram_x_train, y_train)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good defau

In [71]:
ngram_y_pred = balanced_lr_model.predict(ngram_x_test)

In [72]:
print(f"Accuracy: {accuracy_score(y_test, ngram_y_pred)}")
print(classification_report(y_test, ngram_y_pred))
print(confusion_matrix(y_test, ngram_y_pred))

Accuracy: 0.9838565022421525
              precision    recall  f1-score   support

         ham       0.98      1.00      0.99       966
        spam       0.99      0.89      0.94       149

    accuracy                           0.98      1115
   macro avg       0.99      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115

[[965   1]
 [ 17 132]]


#### Let's try for Td-idF + Balanced LR

In [73]:
balanced_tdidf_model = LogisticRegression(class_weight = 'balanced')

In [74]:
balanced_tdidf_model.fit(x_train_tdidf, y_train)

tdidf_y_pred = balanced_tdidf_model.predict(x_test_tdidf)

In [75]:
print(f"Accuracy: {accuracy_score(y_test, tdidf_y_pred)}")
print(classification_report(y_test, tdidf_y_pred))
print(confusion_matrix(y_test, tdidf_y_pred))

Accuracy: 0.9739910313901345
              precision    recall  f1-score   support

         ham       0.99      0.98      0.98       966
        spam       0.89      0.92      0.90       149

    accuracy                           0.97      1115
   macro avg       0.94      0.95      0.94      1115
weighted avg       0.97      0.97      0.97      1115

[[949  17]
 [ 12 137]]


Let's test the probabilities

In [76]:
# to do that we use predict_proba

from sklearn.linear_model import LogisticRegression

balanced_lr_model = LogisticRegression(class_weight = 'balanced')

balanced_lr_model.fit(x_train_bow, y_train)

probabilities = balanced_lr_model.predict_proba(x_test_bow)
print(probabilities[:5])

[[9.88376586e-01 1.16234144e-02]
 [9.99476006e-01 5.23993765e-04]
 [9.98801072e-01 1.19892835e-03]
 [3.54268424e-03 9.96457316e-01]
 [9.94763110e-01 5.23688958e-03]]


Let's create a probabilities model for spam

In [77]:

spam_probabilities = probabilities[:, 1]
print(spam_probabilities[:10])

[1.16234144e-02 5.23993765e-04 1.19892835e-03 9.96457316e-01
 5.23688958e-03 7.15675284e-04 7.90514511e-03 1.83431895e-02
 1.55674027e-05 7.85786692e-03]


#### So from Bow + Balanced LR, our threshold is 13, that is 13/149 * 100 = 8.72%, that is the miss rate.

#### Which means our threshold is 8.72%, that is why we then deploy threshold tuning

#### Since the standard default threshold of LR is 0.5, let's do a 

## Part 6: Threshold Tuning

In [79]:
threshold = 0.5

threshold_y_pred = ['spam' if threshold >= 0.5 else 'ham' for probability in spam_probabilities]

print(classification_report(y_test, threshold_y_pred))

              precision    recall  f1-score   support

         ham       0.00      0.00      0.00       966
        spam       0.13      1.00      0.24       149

    accuracy                           0.13      1115
   macro avg       0.07      0.50      0.12      1115
weighted avg       0.02      0.13      0.03      1115



c:\Users\Michael Ugwumba\OneDrive\Desktop\RESUME PROJECTS\sms-spam-classifier\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Michael Ugwumba\OneDrive\Desktop\RESUME PROJECTS\sms-spam-classifier\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Michael Ugwumba\OneDrive\Desktop\RESUME PROJECTS\sms-spam-classifier\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no pre